In [42]:
import os
import torch
import chromadb
import open_clip
import numpy as np
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv

In [43]:
# Load same model and tokenizer used for embedding images
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, _ = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-H-14")  # the text tokenizer
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [44]:
# embedding text 
@torch.no_grad()
def generate_text_embedding(text: str) -> np.ndarray:
    text = (text or "").strip()   # keep text as-is just strip outer whitespace

    tokens = tokenizer([text]).to(device)          # OpenCLIP tokenizer
    features = model.encode_text(tokens)           # (1, D)

    # same normalization as image embeddings notebook
    features = features / features.norm(dim=-1, keepdim=True)

    return features.cpu().numpy()[0]       

In [45]:
from dotenv import load_dotenv
load_dotenv()

path = Path("D:\\GP\\ECHO\\data\\data\\video_generation\\embeddings\\chroma_db_scripts_landmarks")
path.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=path)

collection = client.get_or_create_collection(
    name="landmarks_scripts",
    metadata={"hnsw:space": "cosine"}
)

In [46]:
import boto3

ACCOUNT_ID   = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY   = os.getenv("R2_ACCESS_KEY")
SECRET_KEY   = os.getenv("R2_SECRET_KEY")
BUCKET_NAME  = os.getenv("R2_BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)
PREFIX = "data/video_generation/outputs/landmarks_scripts/"

objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX)
script_files = [obj['Key'] for obj in objects.get('Contents', []) if obj['Key'].lower().endswith('.txt')]

script_id = 0
for key in script_files:
    obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
    script = obj['Body'].read().decode('utf-8', errors="ignore")

    emb = generate_text_embedding(script)
    
    file_name = key.split('/')[-1]
    landmark_name = Path(file_name).stem

    collection.add(
        ids=[str(script_id)],
        embeddings=[emb],
        metadatas=[{"landmark_name": landmark_name,
                    "path": str(key)}],
    )
    script_id += 1

# initialize chroma_db dir(which images come from cloudflare) then upload the directory to the R2 bucket

In [47]:
count = collection.count()
print("Collection size:", count)

Collection size: 52


In [48]:
result = collection.get(include=["embeddings"], limit=1)

embedding_dim = len(result["embeddings"][0])
print("Embedding size:", embedding_dim)

Embedding size: 1024


In [ ]:
results = collection.get(
    where={"landmark_name": "Famine Stele"}   
)

print("Found:", len(results["ids"]), "documents")

Found: 0 documents


In [ ]:
results = collection.get(
    where={"landmark_name": "Famine Stela"}  
)

print("Found:", len(results["ids"]), "documents")

# 2️⃣ Update each document
for i in range(len(results["ids"])):

    old_meta = results["metadatas"][i]
    new_meta = {
        **old_meta,
        "landmark_name": "Famine Stela",
        "path": old_meta["path"].replace("Famine Stele", "Famine Stela")
    }

    collection.update(
        ids=[results["ids"][i]],
        metadatas=[new_meta]
    )

print("Update completed ✅")

Found: 1 documents
Update completed ✅


In [56]:
results = collection.get(
    where={"landmark_name": "Famine Stela"}  
)

print("Found:", len(results["ids"]), "documents")

Found: 1 documents
